# references

Writes a Table reference for each table your Project reads, from its DESCRIBE.

## Terms

- **Step**: each numbered heading below, with the code cell under it. Its first line says its kind: it changes only files, or only prints; it is a call to the warehouse, yours to run; or it is a decision, yours to make.
- **Names cell**: the code cell after the one that finds the Project. It holds this Workflow's inputs, the start materials you fill in once, each with what to write in its comment; each step's call reads them.
- **Decisions cell**: `DECISIONS`, the code cell after the names cell, where you answer each open Decision a step waits on, by the table's name.
- **Table reference**: a script in references/ describing one table: its columns and their types, its Partitions and its Key.
- **Decision**: one thing about one table that only you decide, such as its location or its Key; the table's files say whether it is answered yet.
- **Key**: the columns whose values name exactly one row of a table.
- **Partition**: the rows of a table that share one value of each column it is partitioned by.
- **Reference sheet**: references/reference_sheet.csv, one table listing every column of every table the Project has a Table reference for.
- **Database module**: a file such as references/ops.py, which gives each table of one database by name, such as ops.jobs.

## Before step 1

Fill in the names cell below and save the notebook: each call reads it, and so do `project.check()`, which names the next step, and `project.do_next()`, the cell before the last.

## Decision rules

- If you run the steps yourself, run each step's cell in turn, in order.
- If you want them run for you, run the last cell, `project.run_workflow(...)`, in place of their cells. It runs every step left in turn, up to the first only you can do, and says where it stopped: go on from that step.
- If you want only the steps that change files run for you, run `project.do_next()`, the cell before the last. It runs them up to the first it can't.

## Self-check

Once the last step is done, each of these holds:

- Each table in `TABLES` has its Table reference in references/, and leaves no Key open.
- references/reference_sheet.csv lists each table's columns.
- `project.check()` names no next step of this Workflow.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import admin, hive_settings, project, run_execute, run_query, show_hive
from warehouse import query, execute

In [ ]:
TABLES = ["ops.job_runs"]  # the tables, by their names in the warehouse, such as ["mart.daily_job_runs"]


In [ ]:
DECISIONS = {"ops.job_runs": {"key": ["run_id"]}}  # run_id holds for each dt, checked on 2026-09-24


## 1. Write the Table references from DESCRIBE

It is a call to the warehouse, yours to run.

- **Goal:** a Table reference for each table in `TABLES`, its columns and their types read from its DESCRIBE.
- **Action:** Run the cell below. It reads each table's DESCRIBE through your query, and writes only files.
- **Done when** each table in `TABLES` has its Table reference in references/; then step 2.

In [ ]:
project.add_references(TABLES, query=query)

## 2. Answer each table's open Decisions in the decisions cell

It is a decision, yours to make.

- **Goal:** an answer to each table's open Key: the columns whose values name exactly one row, or None when no columns do.
- **Action:** Write each answer in the decisions cell, `DECISIONS`, by the table's name as `project.check()` writes it, then save the notebook. To check a Key before you answer it, run `check_key` in a new cell on a Table you make with that Key, since the Table reference's Key is still open. Its `columns=` names each column of the Key and the Partition column, each with None, which leaves its type out. For `run_id` of `ops.job_runs`, partitioned by `dt`, the cell is three lines: `from sqlglot_composer import Table, check_key, equals`; `runs = Table("ops.job_runs", columns={"run_id": None, "dt": None}, partitioned_by=["dt"], key=["run_id"])`; `print(check_key(runs, query=query, where=equals(runs.dt, "2026-09-24")))`. A table with no Partitions leaves out `partitioned_by=` and `where=`. If it says the key holds, then answer it. If it says the key repeats, then add another column to `key=` and `columns=` and check again, or answer None.
- **Done when** the decisions cell answers each Key `project.check()` names for these tables; then step 3.

## 3. Write each answer into its table's Table reference

It changes only files, so project.do_next() runs it for you.

- **Goal:** each answer written into its table's Table reference.
- **Action:** Run the cell below. It names each TODO line the Table reference still holds; no step waits on them, so fill each in by hand. On its `does_not_add_up=[]` line, list each column whose values can't be added up across rows, such as an average, a ratio or a distinct count: `does_not_add_up=["avg_retry_secs"]`. If no column is one, then delete only its `# TODO` comment, and the empty list says no column is one.
- **Done when** each table's Table reference leaves no Key open; then step 4.

In [ ]:
project.write_decisions(DECISIONS)

## 4. Refresh the Reference sheet and the Database modules

It changes only files, so project.do_next() runs it for you.

- **Goal:** each table in the Reference sheet and in its Database module.
- **Action:** Run the cell below. If you change a Table reference after it, such as filling in a TODO line, then run it again: `project.check()` names this step again, saying which Table reference changed.
- **Done when** references/reference_sheet.csv is written again after each table's Table reference last changed; the Workflow is done: go through the Self-check at the top.

In [ ]:
project.refresh_reference_sheet(TABLES)

In [ ]:
project.do_next()

In [ ]:
# Every step left, in place of their cells, up to the first only you can do.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("references", query=query, execute=execute)